In [6]:
import subprocess

# Step 1: Nuke every huggingface_hub installation
subprocess.run("pip uninstall -y huggingface_hub", shell=True)
subprocess.run("pip uninstall -y huggingface_hub", shell=True)  # twice to catch both copies

# Step 2: Remove the stale system copy manually
subprocess.run(
    "rm -rf /usr/local/lib/python3.12/dist-packages/huggingface_hub "
    "/usr/local/lib/python3.12/dist-packages/huggingface_hub-*.dist-info",
    shell=True
)

# Step 3: Fresh install of a known-good version
subprocess.run(
    "pip install -q --no-cache-dir 'huggingface_hub==0.27.1'",
    shell=True
)

# Step 4: Verify before restarting
result = subprocess.run(
    "python -c \""
    "import huggingface_hub; print('version:', huggingface_hub.__version__); "
    "from huggingface_hub import login; print('login import OK'); "
    "from huggingface_hub.utils._auth import _save_stored_tokens; print('_save_stored_tokens OK')"
    "\"",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.stderr.strip():
    print("STDERR:", result.stderr[-800:])

print("\n⚠️  NOW GO TO: Runtime → Restart session")
print("   Then start fresh from CELL 0.")

version: 0.27.1
login import OK
_save_stored_tokens OK


⚠️  NOW GO TO: Runtime → Restart session
   Then start fresh from CELL 0.


In [1]:
import subprocess

def run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# Torch stack
run("pip install -q torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0")

# HF stack — huggingface_hub intentionally NOT pinned, transformers loosely pinned
run("pip install -q 'transformers>=4.41.2,<4.45.0' datasets==2.20.0")
run("pip install -q soundfile librosa")
run("apt-get install -qq -y ffmpeg")

# CLS tokenizer stack
run("pip install -q indic-nlp-library==0.92 indic-unified-parser==1.0.6 "
    "indo-arabic-transliteration==0.1.5 indic-numtowords==1.1.0")
run("pip install -q urduhack==1.1.1")
run("pip install -q git+https://github.com/libindic/indic-trans.git"
    "@0287fa62289968f0ce06cbe2df61cfadf4088c75")
run("pip install -q keras==2.15.0 tensorflow==2.15.0 tensorflow-addons==0.23.0")

# f5-tts + sooktam2 deps
run("pip install -q f5-tts")
run("pip install -q ema-pytorch==0.7.9 x-transformers vocos "
    "torchdiffeq==0.2.5 cached-path")
run("pip install -q safetensors accelerate click==8.0.1 tqdm pydub")

import os
if not os.path.exists("/content/sooktam2"):
    run("git clone https://huggingface.co/bharatgenai/sooktam2 /content/sooktam2")
run("pip install -e /content/sooktam2 --no-cache-dir -q")

# Verify huggingface_hub is still the good version after all installs
result = subprocess.run(
    "python -c \"import huggingface_hub; print('hf_hub version:', huggingface_hub.__version__)\"",
    shell=True, capture_output=True, text=True
)
print(result.stdout.strip())

# Re-apply torchcodec block after pip
import sys, types, importlib.util

def _make_fake_module(name):
    mod = types.ModuleType(name)
    mod.__spec__ = importlib.util.spec_from_loader(name, loader=None)
    mod.__spec__.submodule_search_locations = []
    mod.__loader__ = None
    mod.__path__   = []
    mod.__file__   = None
    mod.__package__ = name
    return mod

for _n in ["torchcodec", "torchcodec.decoders", "datasets.features._torchcodec"]:
    if _n not in sys.modules:
        sys.modules[_n] = _make_fake_module(_n)
sys.modules["datasets.features._torchcodec"].decode = None

import io, numpy as np, soundfile as sf
import datasets.features.audio as _hf_audio

def _soundfile_decode(self, value, token_per_repo_id=None):
    if not self.decode:
        return value
    path = value.get("path"); raw = value.get("bytes")
    arr  = value.get("array"); sr  = value.get("sampling_rate")
    if arr is not None:
        arr = np.array(arr, dtype=np.float32)
        if arr.ndim > 1: arr = arr.mean(axis=1)
        target = self.sampling_rate or sr
        if target and sr and target != sr:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=target)
            sr = target
        return {"path": path, "array": arr, "sampling_rate": sr}
    buf = io.BytesIO(raw) if raw else (io.BytesIO(open(path,"rb").read()) if path else None)
    if buf is None: return value
    arr, sr = sf.read(buf, dtype="float32", always_2d=False)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    target = self.sampling_rate or sr
    if target and target != sr:
        import librosa
        arr = librosa.resample(arr, orig_sr=sr, target_sr=target)
        sr = target
    return {"path": path, "array": arr, "sampling_rate": sr}

_hf_audio.Audio.decode_example = _soundfile_decode
import datasets.features.features as _hf_feat
if hasattr(_hf_feat, "Audio"):
    _hf_feat.Audio.decode_example = _soundfile_decode

print("✅ All deps installed. torchcodec blocked. soundfile patch active.")

hf_hub version: 1.16.1
✅ All deps installed. torchcodec blocked. soundfile patch active.


In [3]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN_HERE"   # ← replace
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in.")

✅ Logged in.


In [4]:
import os, json, random
import numpy as np
import soundfile as sf
from datasets import load_dataset

random.seed(42)
np.random.seed(42)

LANG_CONFIG = "Bengali"    # ← Bengali config
MIN_SAMPLES = 15
TRAIN_SIZE  = 10
TEST_SIZE   = 5
MAX_SCAN    = 20000

for d in [
    "/content/indicvoices_bengali/male/train",
    "/content/indicvoices_bengali/male/test",
    "/content/indicvoices_bengali/female/train",
    "/content/indicvoices_bengali/female/test",
]:
    os.makedirs(d, exist_ok=True)

# Confirm patch active
import datasets.features.audio as _hf_audio, inspect
if "_soundfile_decode" not in str(_hf_audio.Audio.decode_example):
    _hf_audio.Audio.decode_example = _soundfile_decode
    print("✅ Re-applied decode patch.")
else:
    print("✅ Decode patch confirmed active.")

print("Streaming ai4bharat/indicvoices_r [Bengali] train split...")
ds = load_dataset(
    "ai4bharat/indicvoices_r",
    "Bengali",
    split="train",
    streaming=True,
)

speaker_buckets = {}

for i, sample in enumerate(ds):
    if i >= MAX_SCAN:
        print(f"Reached MAX_SCAN={MAX_SCAN}.")
        break

    gender = (sample.get("gender") or "").strip().lower()
    spk_id = (sample.get("speaker_id") or "").strip()
    text   = (sample.get("normalized") or sample.get("text") or "").strip()
    audio  = sample.get("audio")

    if gender not in ("male", "female") or not spk_id or not text or audio is None:
        continue

    arr = audio.get("array")
    sr  = audio.get("sampling_rate")
    if arr is None or sr is None:
        continue

    arr = np.array(arr, dtype=np.float32)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if len(arr) / sr < 1.0:
        continue

    if spk_id not in speaker_buckets:
        speaker_buckets[spk_id] = {"gender": gender, "samples": []}
    speaker_buckets[spk_id]["samples"].append({
        "text": text, "audio_array": arr, "sampling_rate": sr,
    })

    if (i + 1) % 500 == 0:
        m = sum(1 for v in speaker_buckets.values()
                if v["gender"] == "male"   and len(v["samples"]) >= MIN_SAMPLES)
        f = sum(1 for v in speaker_buckets.values()
                if v["gender"] == "female" and len(v["samples"]) >= MIN_SAMPLES)
        print(f"  row {i+1:>6} | qualified male: {m}  female: {f}")

    m_ok = any(v["gender"] == "male"   and len(v["samples"]) >= MIN_SAMPLES
               for v in speaker_buckets.values())
    f_ok = any(v["gender"] == "female" and len(v["samples"]) >= MIN_SAMPLES
               for v in speaker_buckets.values())
    if m_ok and f_ok:
        print(f"✅ Both genders qualified at row {i+1}.")
        break

def pick_best(buckets, gender):
    cands = [(sid, v) for sid, v in buckets.items()
             if v["gender"] == gender and len(v["samples"]) >= MIN_SAMPLES]
    if not cands:
        return None, None
    return sorted(cands, key=lambda x: len(x[1]["samples"]), reverse=True)[0]

male_spk_id,   male_data   = pick_best(speaker_buckets, "male")
female_spk_id, female_data = pick_best(speaker_buckets, "female")

assert male_spk_id,   "❌ No male speaker found. Increase MAX_SCAN."
assert female_spk_id, "❌ No female speaker found. Increase MAX_SCAN."

print(f"\n✅ Male   speaker: {male_spk_id}  ({len(male_data['samples'])} samples)")
print(f"✅ Female speaker: {female_spk_id}  ({len(female_data['samples'])} samples)")

def make_split(data, n_train, n_test):
    s = data["samples"][:]
    random.shuffle(s)
    return s[:n_train], s[n_train: n_train + n_test]

male_train,   male_test   = make_split(male_data,   TRAIN_SIZE, TEST_SIZE)
female_train, female_test = make_split(female_data, TRAIN_SIZE, TEST_SIZE)

print(f"\nMale   — train: {len(male_train)}, test (unseen): {len(male_test)}")
print(f"Female — train: {len(female_train)}, test (unseen): {len(female_test)}")

def save_samples(samples, gender, split, spk_id):
    base = f"/content/indicvoices_bengali/{gender}/{split}"
    os.makedirs(base, exist_ok=True)
    saved = []
    for idx, s in enumerate(samples):
        path = os.path.join(base, f"{spk_id}_{split}_{idx:03d}.wav")
        sf.write(path, s["audio_array"], s["sampling_rate"])
        saved.append({"wav_path": path, "text": s["text"],
                      "speaker_id": spk_id, "sr": s["sampling_rate"]})
    return saved

male_train_saved   = save_samples(male_train,   "male",   "train", male_spk_id)
male_test_saved    = save_samples(male_test,    "male",   "test",  male_spk_id)
female_train_saved = save_samples(female_train, "female", "train", female_spk_id)
female_test_saved  = save_samples(female_test,  "female", "test",  female_spk_id)

manifest = {
    "male_speaker_id": male_spk_id, "female_speaker_id": female_spk_id,
    "male_train": male_train_saved, "male_test": male_test_saved,
    "female_train": female_train_saved, "female_test": female_test_saved,
}
with open("/content/indicvoices_bengali/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n✅ Done. Manifest → /content/indicvoices_bengali/manifest.json")

✅ Decode patch confirmed active.
Streaming ai4bharat/indicvoices_r [Bengali] train split...


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/155 [00:00<?, ?it/s]

  row    500 | qualified male: 0  female: 1
  row   1000 | qualified male: 0  female: 8
✅ Both genders qualified at row 1035.

✅ Male   speaker: S4257950700369146  (15 samples)
✅ Female speaker: S4258795800392027  (30 samples)

Male   — train: 10, test (unseen): 5
Female — train: 10, test (unseen): 5

✅ Done. Manifest → /content/indicvoices_bengali/manifest.json


In [5]:
import librosa
import numpy as np

def pick_reference(train_saved):
    scored = []
    for s in train_saved:
        y, sr = librosa.load(s["wav_path"], sr=None)
        dur = len(y) / sr
        score = abs(dur - 6.0)
        scored.append((score, dur, s))
    scored.sort(key=lambda x: x[0])
    _, dur, best = scored[0]
    return best, dur

male_ref,   male_ref_dur   = pick_reference(male_train_saved)
female_ref, female_ref_dur = pick_reference(female_train_saved)

print(f"Male   reference: {male_ref['wav_path']}")
print(f"  text: {male_ref['text']}")
print(f"  dur : {male_ref_dur:.2f}s\n")

print(f"Female reference: {female_ref['wav_path']}")
print(f"  text: {female_ref['text']}")
print(f"  dur : {female_ref_dur:.2f}s")

MALE_REF_WAV    = male_ref["wav_path"]
MALE_REF_TEXT   = male_ref["text"]
FEMALE_REF_WAV  = female_ref["wav_path"]
FEMALE_REF_TEXT = female_ref["text"]

Male   reference: /content/indicvoices_bengali/male/train/S4257950700369146_train_003.wav
  text: সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে
  dur : 6.81s

Female reference: /content/indicvoices_bengali/female/train/S4258795800392027_train_004.wav
  text: এটাও খুব সহজে আমরা পেয়ে গেছি যেন মনে হবে
  dur : 6.21s


In [6]:
import sys, os

sys.path.insert(0, "/content/sooktam2/src")
sys.path.insert(0, "/content/sooktam2")

# Ensure f5_tts is available
try:
    import f5_tts
    print(f"✅ f5_tts at: {f5_tts.__file__}")
except ImportError:
    import subprocess
    subprocess.run("pip install -q f5-tts", shell=True)
    import f5_tts
    print(f"✅ f5_tts installed and imported.")

from transformers import AutoModel

print("Loading bharatgenai/sooktam2...")
tts_model = AutoModel.from_pretrained(
    "bharatgenai/sooktam2",
    trust_remote_code=True,
)
print("✅ sooktam2 loaded.")

✅ f5_tts at: None
Loading bharatgenai/sooktam2...


config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

hf_auto.py:   0%|          | 0.00/463 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/bharatgenai/sooktam2:
- hf_auto.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWar

model_1250000.pt:   0%|          | 0.00/5.38G [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/54.4M [00:00<?, ?B/s]


vocab :  /root/.cache/huggingface/hub/models--bharatgenai--sooktam2/snapshots/eb270c0ceebd58e0e513820eb71c637e9f486a7b/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--bharatgenai--sooktam2/snapshots/eb270c0ceebd58e0e513820eb71c637e9f486a7b/model_1250000.pt 

✅ sooktam2 loaded.


In [7]:
import os, json, shutil

OUTPUT_ROOT = "/content/outputs_bengali"

def generate_for_split(split_saved, ref_wav, ref_text, gender, split_name):
    out_dir = os.path.join(OUTPUT_ROOT, gender, split_name)
    os.makedirs(out_dir, exist_ok=True)
    results = []
    for idx, sample in enumerate(split_saved):
        out_gt  = os.path.join(out_dir, f"gt_{idx:03d}.wav")
        out_gen = os.path.join(out_dir, f"gen_{idx:03d}.wav")
        shutil.copy(sample["wav_path"], out_gt)
        try:
            wav, sr, _ = tts_model.infer(
                ref_file     = ref_wav,
                ref_text     = ref_text,
                gen_text     = sample["text"],
                tokenizer    = "cls",
                cls_language = "bengali",      # ← Bengali
                file_wave    = out_gen,
            )
            status = "ok"
            print(f"  [{gender}/{split_name}] {idx+1}/{len(split_saved)} ✅")
        except Exception as e:
            status = f"error: {e}"
            print(f"  [{gender}/{split_name}] {idx+1}/{len(split_saved)} ❌ {e}")
        results.append({"idx": idx, "text": sample["text"],
                        "gt_path": out_gt, "gen_path": out_gen, "status": status})
    return results

print("=== Male train ===")
male_train_results   = generate_for_split(male_train_saved,   MALE_REF_WAV,   MALE_REF_TEXT,   "male",   "train")
print("\n=== Male test (unseen) ===")
male_test_results    = generate_for_split(male_test_saved,    MALE_REF_WAV,   MALE_REF_TEXT,   "male",   "test")
print("\n=== Female train ===")
female_train_results = generate_for_split(female_train_saved, FEMALE_REF_WAV, FEMALE_REF_TEXT, "female", "train")
print("\n=== Female test (unseen) ===")
female_test_results  = generate_for_split(female_test_saved,  FEMALE_REF_WAV, FEMALE_REF_TEXT, "female", "test")

with open(os.path.join(OUTPUT_ROOT, "split_results.json"), "w", encoding="utf-8") as f:
    json.dump({"male_train": male_train_results, "male_test": male_test_results,
               "female_train": female_train_results, "female_test": female_test_results},
              f, ensure_ascii=False, indent=2)

print("\n✅ Train/test generation done.")

=== Male train ===
Converting audio...
Using custom reference text...

ref_text   সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে. 
gen_text 0 কারণ আমি এরকম অসুবিধার অনেকবার সম্মুখীন হয়েছি একবার আমাদের এই গ্রামে এক অনুষ্ঠান বাড়িতে যখন বিয়েবাড়ির অনুষ্ঠানে গিয়েছিলাম


Generating audio in 1 batches...


100%|██████████| 1/1 [00:21<00:00, 21.63s/it]


  [male/train] 1/10 ✅
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে. 
gen_text 0 সে হাসপাতালে পাঠানোর পরে আমরা সমস্ত রকম সুবিধা পেয়েছিলাম কিন্তু আমাদের এই হাসপাতালে সমস্ত রকম সুবিধা থাকে না


Generating audio in 1 batches...


100%|██████████| 1/1 [00:17<00:00, 17.41s/it]


  [male/train] 2/10 ✅
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে. 
gen_text 0 কারণ আমি যখন তার ছটার পর থেকে দু ঘন্টা আটটা অবধি যখন দৌড়ই সেই ফুটবল গ্রাউণ্ডে কোনওরকম ভয় থাকে না


Generating audio in 1 batches...


...

  [female/test] 3/5 ✅
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   এটাও খুব সহজে আমরা পেয়ে গেছি যেন মনে হবে. 
gen_text 0 আমরা কোনো কিছু পড়াশোনার ক্ষেত্রেও যদি না বুঝতে পারি সেখানে


Generating audio in 1 batches...


100%|██████████| 1/1 [00:18<00:00, 18.07s/it]


  [female/test] 4/5 ✅
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   এটাও খুব সহজে আমরা পেয়ে গেছি যেন মনে হবে. 
gen_text 0 স্মার্ট ফোন আসার পরে আমাদের জীবনে অনেক উন্নতি ঘটেছে আমরা যেমন পৃথিবীর সব কিছু হাতের মুঠোয় যেন পেয়ে গেছি কারণ স্মার্ট ফোন আমাদের হাতে থাকার ফলে আমরা হয়তো বাইরে যাবো কোথাও


Generating audio in 1 batches...


100%|██████████| 1/1 [00:46<00:00, 46.77s/it]

  [female/test] 5/5 ✅

✅ Train/test generation done.


In [9]:
import json

# Upload bengali_evaluation_set.json to Colab first, then run this cell
EVAL_SET_PATH = "/content/bengali_evaluation_set.json"

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Structure: list of dicts with "bengali_sentence" as the text key
eval_set = raw[:20]
TEXT_KEY  = "bengali_sentence"

print(f"✅ Loaded {len(eval_set)} Bengali evaluation sentences.\n")
for i, item in enumerate(eval_set):
    print(f"  [{i+1:02d}] [{item['benchmark_category'].split(':')[0]}]")
    print(f"       {item[TEXT_KEY][:90]}")

✅ Loaded 20 Bengali evaluation sentences.

  [01] [Intelligibility (SUS)]
       অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে।
  [02] [Intelligibility (SUS)]
       ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হলো।
  [03] [Intelligibility (SUS)]
       শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে।
  [04] [Prosody & Phrasing]
       বর্ষার শেষে মেঘলা আকাশে হঠাৎ এক ঝাঁক সাদা বক আর সবুজ ঘাসের ওপর ব্যাঙের ডাক শোনা গেল।
  [05] [Naturalness]
       সে রেগে গিয়ে বলল, "আমার ছাতা আর ফলের ঝুড়িটা কোথায় জেনেশুনে লুকিয়ে রেখেছ?"
  [06] [Robustness (Code-Mixing)]
       জুম মিটিং চলার সময় ল্যাপটপের ব্রাইটনেস কমিয়ে পিডিএফ ফাইলটা আমাকে ফরোয়ার্ড করো।
  [07] [Robustness (Code-Mixing)]
       ট্রেনের ড্রাইভার ব্লুটুথ হেডফোন কানে দিয়ে স্ক্রিনের দিকে তাকিয়ে ইমার্জেন্সি ব্রেক কষলেন।
  [08] [Text Normalization]
       ২০০৫ সালের ১৫ই আগস্ট, কলকাতার তাপমাত্রা ছিল ঠিক ৪২.৫ ডিগ্রি সেলসিয়াস।
  [09] [Text Normalization]
       পৌনে ত

In [10]:
import os, json

EVAL_MALE_DIR   = "/content/outputs_bengali/eval_male"
EVAL_FEMALE_DIR = "/content/outputs_bengali/eval_female"
os.makedirs(EVAL_MALE_DIR,   exist_ok=True)
os.makedirs(EVAL_FEMALE_DIR, exist_ok=True)

eval_sentences = [item[TEXT_KEY] for item in eval_set]

def generate_eval(sentences, ref_wav, ref_text, out_dir, gender_label):
    results = []
    print(f"\n=== Generating EVAL — {gender_label.upper()} ===")
    for idx, text in enumerate(sentences):
        out_path = os.path.join(out_dir, f"eval_{idx+1:03d}.wav")
        try:
            wav, sr, _ = tts_model.infer(
                ref_file     = ref_wav,
                ref_text     = ref_text,
                gen_text     = text,
                tokenizer    = "cls",
                cls_language = "bengali",      # ← Bengali
                file_wave    = out_path,
            )
            status = "ok"
            print(f"  [{idx+1}/20] ✅  {os.path.basename(out_path)}")
        except Exception as e:
            status = f"error: {e}"
            print(f"  [{idx+1}/20] ❌  {e}")
        results.append({"idx": idx+1, "text": text,
                        "category": eval_set[idx]["benchmark_category"],
                        "wav_path": out_path, "status": status})
    return results

male_eval_results   = generate_eval(eval_sentences, MALE_REF_WAV,   MALE_REF_TEXT,   EVAL_MALE_DIR,   "male")
female_eval_results = generate_eval(eval_sentences, FEMALE_REF_WAV, FEMALE_REF_TEXT, EVAL_FEMALE_DIR, "female")

eval_manifest = {
    "language": "Bengali",
    "male_speaker_id":   male_ref["speaker_id"],
    "female_speaker_id": female_ref["speaker_id"],
    "male_ref_wav":      MALE_REF_WAV,
    "female_ref_wav":    FEMALE_REF_WAV,
    "male_eval":   male_eval_results,
    "female_eval": female_eval_results,
}
with open("/content/outputs_bengali/eval_manifest.json", "w", encoding="utf-8") as f:
    json.dump(eval_manifest, f, ensure_ascii=False, indent=2)

print(f"\n✅ All 20 Bengali eval speeches generated.")
print(f"   Male   → {EVAL_MALE_DIR}")
print(f"   Female → {EVAL_FEMALE_DIR}")


=== Generating EVAL — MALE ===
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে. 
gen_text 0 অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:17<00:00, 17.58s/it]


  [1/20] ✅  eval_001.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে. 
gen_text 0 ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হলো।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:15<00:00, 15.99s/it]


  [2/20] ✅  eval_002.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   সাড়ে পাঁচটা অবধি আমি রাস্তায় দৌড়ই কারণ রাস্তায় অনেক যানবাহন চলাচল হয় সেই কারণে. 
gen_text 0 শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে।


Generating audio in 1 batches...


...

  [18/20] ✅  eval_018.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   এটাও খুব সহজে আমরা পেয়ে গেছি যেন মনে হবে. 
gen_text 0 হঠাৎ উৎসবের দিনে নদীর জল এতটাই বেড়ে গেল যে গ্রামের সবাই খুব ভয় পেয়ে গেল।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:21<00:00, 21.68s/it]


  [19/20] ✅  eval_019.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   এটাও খুব সহজে আমরা পেয়ে গেছি যেন মনে হবে. 
gen_text 0 নৌকো করে নদীর থৈথৈ জলে ভাসার সময়, উনুনের মিষ্টি ধোঁয়ায় তার চোখ বুজে এল।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:21<00:00, 21.33s/it]

  [20/20] ✅  eval_020.wav

✅ All 20 Bengali eval speeches generated.
   Male   → /content/outputs_bengali/eval_male
   Female → /content/outputs_bengali/eval_female


In [11]:
import shutil, os
from google.colab import files

# ── Choose which output folder to zip ──
# Change this to /content/outputs  if you're downloading Hindi outputs
OUTPUT_DIR = "/content/outputs_bengali"

# Confirm folder exists and show contents summary
if not os.path.exists(OUTPUT_DIR):
    print(f"❌ Folder not found: {OUTPUT_DIR}")
else:
    total_files = sum(len(fs) for _, _, fs in os.walk(OUTPUT_DIR))
    total_size  = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, fs in os.walk(OUTPUT_DIR) for f in fs
    )
    print(f"📁 {OUTPUT_DIR}")
    print(f"   Total files : {total_files}")
    print(f"   Total size  : {total_size / 1024 / 1024:.1f} MB")
    print("\nFolder structure:")
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        dirs.sort()
        level  = root.replace(OUTPUT_DIR, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in sorted(fnames):
            fpath = os.path.join(root, f)
            kb    = os.path.getsize(fpath) / 1024
            print(f"{indent}  {f}  ({kb:.1f} KB)")

    # ── Zip and download ──
    zip_path = OUTPUT_DIR.rstrip("/") + "_download"
    print(f"\nZipping to {zip_path}.zip ...")
    shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

    zip_size = os.path.getsize(f"{zip_path}.zip") / 1024 / 1024
    print(f"✅ Archive ready: {zip_path}.zip  ({zip_size:.1f} MB)")
    print("Starting download...")
    files.download(f"{zip_path}.zip")

📁 /content/outputs_bengali
   Total files : 102
   Total size  : 55.5 MB

Folder structure:
outputs_bengali/
  eval_manifest.json  (17.9 KB)
  split_results.json  (13.2 KB)
  eval_female/
    eval_001.wav  (458.0 KB)
    eval_002.wav  (506.5 KB)
    eval_003.wav  (590.5 KB)
    eval_004.wav  (593.0 KB)
    eval_005.wav  (528.0 KB)
    eval_006.wav  (587.5 KB)
    eval_007.wav  (660.5 KB)
    eval_008.wav  (501.0 KB)
    eval_009.wav  (531.0 KB)
    eval_010.wav  (525.5 KB)
    eval_011.wav  (571.5 KB)
    eval_012.wav  (676.5 KB)
    eval_013.wav  (477.0 KB)
    eval_014.wav  (652.5 KB)
    eval_015.wav  (601.0 KB)
    eval_016.wav  (625.5 KB)
    eval_017.wav  (487.5 KB)
    eval_018.wav  (617.5 KB)
    eval_019.wav  (531.0 KB)
    eval_020.wav  (514.5 KB)
  eval_male/
    eval_001.wav  (257.5 KB)
    eval_002.wav  (285.0 KB)
    eval_003.wav  (332.0 KB)
    eval_004.wav  (333.5 KB)
    eval_005.wav  (297.0 KB)
    eval_006.wav  (330.5 KB)
    eval_007.wav  (371.5 KB)
    eval_008.wav

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>